# Hands-on 2 — Same area, second product: TESSERA

**Chapter 3 · Where and how to access embeddings**

AlphaEarth is not the only global, pixel-level embedding product. TESSERA, from the University of Cambridge, gives 128-dimensional embeddings at 10 m for every year since 2017 in Europe, and is released under CC0. It has its own tiling, its own storage format and its own Python library, `geotessera`. In this notebook you will:

1. fetch TESSERA tiles for the East Attica study area,
2. put them on the AlphaEarth pixel grid so the two products can be compared pixel by pixel,
3. compare what each product "sees": PCA pictures, similarity maps from the same reference point, and a measure of how much the two embedding spaces agree,
4. learn what to watch for when you move between products: vector norms, nodata conventions, tile borders.

Requires the cache written by Hands-on 1 (`data/cache/aef_attica_2024.npz`). Expected run time about 3 minutes; ~60 MB downloaded.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from affine import Affine
from rasterio.crs import CRS

import eoemb

AOI = (23.90, 38.00, 24.00, 38.10)
cache = eoemb.DATA_DIR / "cache"

## 1. Reload AlphaEarth from the cache and look at the area in true colour

The course ships a cloud-free Sentinel-2 image of the area (13 July 2024) already resampled onto the AlphaEarth grid. Keep it open in a corner of your mind: everything we compute should make sense against it.

In [ ]:
z = np.load(cache / "aef_attica_2024.npz")
transform = Affine.from_gdal(*z["transform"])
crs = CRS.from_string(str(z["crs"]))
aef = eoemb.dequantize(z["raw"])                     # (64, H, W), unit vectors, NaN = nodata
H, W = aef.shape[1:]
extent = eoemb.lonlat_extent(transform, crs, (H, W))

with rasterio.open(eoemb.DATA_DIR / "s2_attica_20240713_rgb.tif") as src:
    s2_rgb = np.moveaxis(src.read(), 0, -1)
    assert src.transform == transform and src.shape == (H, W), "S2 image must share the AlphaEarth grid"

plt.figure(figsize=(8, 8)); plt.imshow(s2_rgb, extent=extent); plt.title("Sentinel-2 true colour, 13 July 2024");

## 2. Fetch TESSERA

TESSERA is distributed as **0.1° × 0.1° tiles**, each stored as a NumPy array in the UTM zone of the tile. The `geotessera` library keeps a registry of about 4.4 million tiles, downloads what you ask for and de-quantizes it (the tiles on disk are int8 plus a per-pixel scale factor, the same storage trick as AlphaEarth, done slightly differently).

First, which tiles do we need?

In [ ]:
from geotessera import GeoTessera

gt = GeoTessera(embeddings_dir=str(cache / "tessera"))
tiles = gt.registry.load_blocks_for_region(bounds=AOI, year=2024)
print(len(tiles), "tiles returned for the bounding box (the registry pads with neighbours):")
print(tiles)

Our box is exactly the tile centred on (23.95, 38.05). Fetch just that one and inspect it.

In [ ]:
t0 = time.time()
year, tlon, tlat, tile, tile_crs, tile_tr = next(gt.fetch_embeddings([(2024, 23.95, 38.05)]))
print(f"fetched in {time.time() - t0:.1f} s")
print("shape", tile.shape, tile.dtype, "→ (rows, cols, 128): TESSERA puts the embedding axis LAST")
print("CRS", tile_crs, "\ntransform", tile_tr)

norms = np.linalg.norm(tile, axis=-1)
print("vector norm min/median/max:", norms.min().round(1), np.median(norms).round(1), norms.max().round(1))

Three differences from AlphaEarth already:

| | AlphaEarth | TESSERA |
|---|---|---|
| array layout | `(64, rows, cols)` | `(rows, cols, 128)` |
| vector norm | exactly 1 | around 30, varies per pixel |
| pixel grid | 10 m, origin on the Earth Engine tile grid | 10 m, origin on the 0.1° tile corner |

The norm matters: with TESSERA you cannot use a plain dot product as cosine similarity, you have to normalise first. The grid matters: the two products' pixels do not line up, they are offset by a fraction of a pixel and TESSERA's rows/columns count differ by one. To compare pixel to pixel we resample TESSERA onto the AlphaEarth grid with nearest neighbour (no values are invented, each output pixel takes the value of the closest TESSERA pixel).

`eoemb.tessera_read_like` does the tile selection, the fetch and the resampling in one go:

In [ ]:
t0 = time.time()
tes = eoemb.tessera_read_like(AOI, 2024, transform, crs, (H, W), embeddings_dir=cache / "tessera")
print(f"TESSERA on the AlphaEarth grid: {tes.shape} in {time.time() - t0:.1f} s; NaN fraction {np.isnan(tes[0]).mean():.4f}")

tes_unit = tes / np.linalg.norm(tes, axis=0, keepdims=True)   # unit vectors for cosine similarity

## 3. Two pictures of the same place

In [ ]:
rgb_aef = eoemb.pca_rgb(aef)
rgb_tes = eoemb.pca_rgb(tes)

fig, ax = plt.subplots(1, 3, figsize=(18, 6.5))
ax[0].imshow(s2_rgb, extent=extent); ax[0].set_title("Sentinel-2 true colour")
ax[1].imshow(rgb_aef, extent=extent); ax[1].set_title("AlphaEarth 2024 · PCA(3)")
ax[2].imshow(rgb_tes, extent=extent); ax[2].set_title("TESSERA 2024 · PCA(3)")
plt.tight_layout()

Same year, same pixels, two very different pictures. In TESSERA's first three components the Penteli forest, the built-up coast and the farmland come out as three clear colour groups, and the sea has a value like any other surface. In AlphaEarth the first components follow relief and the coastline, and the sea is masked out.

Neither is "better" from this picture alone. PCA shows the *largest* sources of variation, and the two models were trained with different inputs (TESSERA: Sentinel-1 and Sentinel-2 time series only; AlphaEarth: also elevation, climate, and more), so the largest variation is not the same thing. What matters for your work is whether the embedding separates the classes *you* care about, and that is a question for a classifier (Hands-on 4), not for a PCA image.

## 4. Same question, both products: "what is similar to this pixel?"

In [ ]:
points = pd.DataFrame({
    "name": ["forest", "town", "burned 2018", "fields"],
    "lon": [23.935, 23.9765, 23.985, 23.96],
    "lat": [38.085, 38.0855, 38.045, 38.005],
})
rows, cols = eoemb.lonlat_to_rowcol(points.lon, points.lat, transform, crs)

fig, ax = plt.subplots(2, 4, figsize=(20, 9))
for j, name in enumerate(points.name):
    for i, (label, cube) in enumerate([("AlphaEarth", aef), ("TESSERA", tes_unit)]):
        q = cube[:, rows[j], cols[j]]
        sim = eoemb.cosine_similarity_map(cube, q)
        im = ax[i, j].imshow(sim, extent=extent, vmin=0.5, vmax=1, cmap="viridis")
        ax[i, j].plot(points.lon[j], points.lat[j], "r+", ms=14, mew=2)
        ax[i, j].set_title(f"{label} · similar to '{name}'")
plt.tight_layout()

Look column by column. Both products agree on the broad answer: forest lights up the forest, town lights up the settlements, fields light up the coastal plain. But the *shape* of the similarity values differs. In AlphaEarth similarity decays gradually: almost every land pixel is above 0.5 and you can see the relief again in the forest map. TESSERA is more decisive: pixels of the same kind score very high, everything else drops off quickly, down to individual built-up pixels between gardens in the town map. This is a property of how each embedding space is shaped, and it means **a similarity threshold that works for one product will not transfer to the other** (exercise E3 puts numbers on this).

## 5. Do the two embedding spaces agree?

A more careful way to compare: take a random sample of pixels, compute all pairwise cosine distances within AlphaEarth and within TESSERA, and correlate the two distance lists. If two pixels are close in one space, are they close in the other?

In [ ]:
from scipy.stats import spearmanr

rng = np.random.default_rng(0)
valid = np.flatnonzero(~np.isnan(aef[0]).ravel() & ~np.isnan(tes[0]).ravel())
idx = rng.choice(valid, 2000, replace=False)

A = aef.reshape(64, -1)[:, idx].T          # (2000, 64)
T = tes_unit.reshape(128, -1)[:, idx].T    # (2000, 128)
dA = 1 - A @ A.T
dT = 1 - T @ T.T
iu = np.triu_indices(len(idx), k=1)
rho = spearmanr(dA[iu], dT[iu]).statistic
print(f"Spearman correlation of pairwise distances: {rho:.3f}")

plt.figure(figsize=(5, 5))
plt.hexbin(dA[iu], dT[iu], gridsize=60, bins="log", cmap="Blues")
plt.xlabel("cosine distance in AlphaEarth"); plt.ylabel("cosine distance in TESSERA"); plt.title(f"2 000 pixels, ρ = {rho:.2f}");

A clearly positive correlation (around 0.6): the two products broadly agree on *which* places resemble each other, even though they were built by different teams from different inputs. That is reassuring, and it is what makes embeddings useful as a general-purpose description of the land surface rather than a quirk of one model. The agreement is far from perfect, though. The scatter is where the products disagree, and the plot also shows the different scales: AlphaEarth distances are compressed towards zero, TESSERA spreads them out. Before choosing a product for a specific task you would test both on that task, which is what Hands-on 4 does for land cover.

## 6. Nodata and the sea

AlphaEarth masks the sea (nodata), TESSERA does not. Neither is wrong, but if you switch products your code must not assume one behaviour.

In [ ]:
sea_aef = np.isnan(aef[0])
sea_ref = tes_unit[:, *eoemb.lonlat_to_rowcol([23.995], [38.09], transform, crs)].ravel()
sea_tes = eoemb.cosine_similarity_map(tes_unit, sea_ref) > 0.95

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(sea_aef, extent=extent, cmap="gray_r"); ax[0].set_title("AlphaEarth: nodata (sea + footprint edge)")
ax[1].imshow(sea_tes, extent=extent, cmap="gray_r"); ax[1].set_title("TESSERA: pixels similar to a sea pixel (> 0.95)")
plt.tight_layout()

## 7. A note on caching

We do not save the aligned TESSERA cube: at 128 × 1138 × 913 float32 it is 530 MB, five times the AlphaEarth cube (int8 storage is not a detail!). `geotessera` keeps the downloaded tiles in `data/cache/tessera`, so re-running `tessera_read_like` later takes ~15 s and no network. Hands-on 4 does exactly that.

---
## Exercises

**E1.** `gt.fetch_embedding(lon, lat, year)` fetches the single tile containing a point. Fetch the 2019 tile for our area and compute, per pixel, the cosine distance between TESSERA 2019 and TESSERA 2024 (remember to normalise). Where does the map light up? Compare with your knowledge of what happened here in 2018 and in August 2024.

**E2.** Repeat section 5 with 128-dimensional TESSERA truncated to its first 32 axes. Does the agreement with AlphaEarth drop a lot? (TESSERA v2 is trained so that truncated vectors remain useful, so-called Matryoshka embeddings; the v1 tiles served today are not, which makes this a fair test of what truncation costs.)

**E3.** Pick the "town" reference point. Count how many pixels have similarity > 0.9 to it in AlphaEarth and in TESSERA. Which threshold in TESSERA gives about the same number of pixels as 0.9 in AlphaEarth?